# Dense-Sparse DTW

The goal of this notebook is to align files using Dense Sparse DTW

In [1]:
%matplotlib inline

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import librosa as lb
import os.path
from pathlib import Path
import pickle
import multiprocessing
import time
import gc
import seaborn as sns
import glob
import pandas as pd
import time

Align with Dense-Sparse DTW

In [3]:
def selectFeatures(F, frac_keep):
    '''
    Selects a subset of features that have the highest flux.

    Inputs:
        F: feature matrix of size DxN, where D is the feature dimension and N is the number of features
        frac_keep: the fraction of features to keep, a scalar between 0 and 1

    Returns:
        F_sel: a DxM feature matrix containing the selected subset of features
        idx_sel: an array of length M specifying the indices of the features that were selected
        gaplens: an array of length M specifying the gap lengths between selected features
        flux_thresh: the threshold used to select features based on their flux
    '''
    flux_vals = np.sum(np.abs(F[:,0:-1] - F[:,1:]), axis=0)
    flux_thresh = sorted(flux_vals, reverse=True)[int(np.round(frac_keep * len(flux_vals)))-1]
    idx_sel = np.where(np.array(flux_vals > flux_thresh) == 1)[0]
    gaplens = idx_sel[1:] - idx_sel[0:-1]
    gaplens = np.append(gaplens, len(flux_vals) - idx_sel[-1])
    F_sel = F[:,idx_sel]
    
    return F_sel, idx_sel, gaplens, flux_thresh


# @jit(nopython=True)
def dtw_sparse_subseq(C, gaplens, steps=np.array([1,1,1,2,2,1]).reshape((-1,2)), weights = np.array([1,1,2])):
    '''
    A variant of subsequence DTW that aligns a selected subset of query features against a longer reference sequence.
    The query sequence can start and end anywhere in the reference sequence, and the alignment handles gaps between
    selected query features.
    
    Inputs:
        C: an MxN matrix of pairwise costs, where M is the length of the (selected) query features and N is the length of
           the reference sequence
        gaplens: an array of length M specifying the gap lengths between selected features

    Returns:
        D: cumulative cost matrix, size MxN
        B: backtrace matrix of size MxN, each element specifies either the step index (if dense matching)
           or the number of reference frames skipped (if sparse matching)
        path: a numpy array of (row, col) coordinates for the optimal path
    '''
    D = np.ones(C.shape) * np.inf
    B = np.zeros(C.shape, dtype=np.int32)
    # steps = np.array([1,1,1,2,2,1]).reshape((-1,2))
    # weights = np.array([1,1,2])

    D[0, :] = C[0,:]

    for row in range(1, C.shape[0]):
        for col in range(1, C.shape[1]):
            
            if row >= 2 and gaplens[row-2] == 1 and gaplens[row-1] == 1:
                
                # dense matching
                bestCost = D[row, col]
                bestCostIndex = -1
                for stepIndex in range(steps.shape[0]):
                    if row - steps[stepIndex][0] >= 0 and col - steps[stepIndex][1] >= 0:
                        costForStep = C[row, col] * weights[stepIndex] + D[row - steps[stepIndex][0], col - steps[stepIndex][1]]
                        if costForStep < bestCost:
                            bestCost = costForStep
                            bestCostIndex = stepIndex
                D[row, col] = bestCost
                B[row, col] = bestCostIndex
                
            else:
                
                # sparse matching
                # cstep_lbound = int(np.ceil(gaplens[row-1]/2))
                # cstep_ubound = gaplens[row-1]*2 + 1
                # bestCost = D[row, col]
                # for cstep in range(cstep_lbound, cstep_ubound):
                #     rprev = row - 1
                #     cprev = col - cstep
                #     if cprev >= 0:
                #         costForStep = C[row, col] + D[rprev, cprev]
                #         if costForStep < bestCost:
                #             bestCost = costForStep
                #             bestCostIndex = cstep
                # D[row, col] = bestCost
                # B[row, col] = bestCostIndex

                crange_lbound = max(col - gaplens[row-1]*2, 0)
                crange_ubound = col - int(np.ceil(gaplens[row-1]/2)) + 1
                #if crange_lbound >= crange_ubound:
                #    print(f'crange_lb = {crange_lbound}, crange_ubound = {crange_ubound}, row = {row}, col = {col}')
                if crange_ubound > crange_lbound:
                    D[row, col] = np.min(D[row-1, crange_lbound:crange_ubound]) + C[row,col]
                    B[row, col] = col - (crange_lbound + np.argmin(D[row-1, crange_lbound:crange_ubound]))
    
    path = dtw_backtrace_sparse(D, B, gaplens, steps, subseq=True)
    path.reverse()
    path = np.array(path).T

    return D, B, path



# @jit(nopython=True)
def dtw_backtrace_sparse(D, B, gaplens, steps, subseq):
    '''
    Backtraces through the cumulative cost matrix D
    
    Inputs:
        D: cumulative cost matrix
        B: backtrace matrix
        gaplens: array specifying the gap lengths between selected features
        steps: a numpy matrix specifying the allowable transitions.  It should be of dimension (L, 2), where each row specifies (row step, col step)
        subseq: boolean indicating whether to assume a subsequence alignment
    
    Returns:
        A numpy array of (row, col) coordinates for the optimal path.
    '''

    rstart = B.shape[0] - 1
    if subseq:
        cstart = np.argmin(D[-1])
    else:
        cstart = B.shape[1] - 1
    pos = (rstart, cstart)
    path = []
    path.append(pos)
    while (pos[0] != 0 and pos[1] != 0) or (pos[0] and subseq):
        
        (row, col) = pos
        if row >= 2 and gaplens[row-1] == 1 and gaplens[row-2] == 1:
            
            # dense matching
            stepidx = B[row, col]
            (rstep, cstep) = steps[stepidx]
            pos = (row-rstep, col-cstep)
            path.append(pos)
            
        else:
            
            # sparse matching
            rstep = 1
            cstep = B[row, col]
            pos = (row-rstep, col-cstep)
            path.append(pos)

    return path

In [4]:
def alignDenseSparseDTW(featfile1, featfile2, steps, weights, downsample, frac_keep=0.5, outfile=None, profile=False):
    F1 = np.load(featfile1) # 12 x N query
    F2 = np.load(featfile2) # 12 x M
    
    # check valid path
    if max(F1.shape[1], F2.shape[1]) / min(F1.shape[1], F2.shape[1]) >= 2:
        if outfile is not None:
            with open(outfile, "wb") as f:
                pickle.dump(None, f)
        return None
    
    times = []
    times.append(time.time())
    
    # downsample first, to stay on same frame grid as standard DTW
    F1_ds = F1[:, 0::downsample]
    F2_ds = F2[:, 0::downsample]
    times.append(time.time())
    
    # feature selection
    F1_sel, idx_sel, gaplens, flux_thresh = selectFeatures(F1_ds, frac_keep)
    times.append(time.time())
    
    # calculate pairwise cost matrix
    C = 1 - F1_sel.T @ F2_ds
    D, B, wp = dtw_sparse_subseq(C, gaplens, steps, weights)
    wp[0,:] = idx_sel[wp[0,:]] # convert back to O frames
    t_end = time.time()
    

    if outfile is not None:
        with open(outfile, "wb") as f:
            pickle.dump(wp, f)

    if profile:
        return wp, np.diff(times)
    return wp
    
    
    

In [5]:
def alignDenseSparseDTW_batch(querylist, featdir1, featdir2, outdir, n_cores, steps, weights, downsample, frac_keep=0.5):
    
    outdir.mkdir(parents=True, exist_ok=True)
    
    # prep inputs for parallelization
    inputs = []
    with open(querylist, 'r') as f:
        for line in f:
            parts = line.strip().split(' ')
            assert len(parts) == 2
            featfile1 = (featdir1 / parts[0]).with_suffix('.npy')
            featfile2 = (featdir2 / parts[1]).with_suffix('.npy')
            queryid = os.path.basename(parts[0]) + '__' + os.path.basename(parts[1])
            outfile = (outdir / queryid).with_suffix('.pkl')
            if os.path.exists(outfile):
                print(f"Skipping {outfile}")
            else:
                inputs.append((featfile1, featfile2, steps, weights, downsample, frac_keep, outfile))

    # process files in parallel
    pool = multiprocessing.Pool(processes = n_cores)
    pool.starmap(alignDenseSparseDTW, inputs)
    
    return

In [6]:
featfile1 = 'features/clean/Chopin_Op068No3/Chopin_Op068No3_Tomsic-1995_pid9190-11.npy'
featfile2 = 'features/clean/Chopin_Op068No3/Chopin_Op068No3_Cortot-1951_pid9066b-19.npy'
steps = np.array([1,1,1,2,2,1]).reshape((-1,2))
weights = np.array([2,3,3])
downsample = 1
wp = alignDenseSparseDTW(featfile1, featfile2, steps, weights, downsample)

In [7]:
query_list = 'cfg_files/query.test.list'
featdir1 = Path('features/clean')
featdir2 = Path('features/clean') # in case you want to align clean vs noisy
outdir = Path('experiments_test/dsdtw_clean')
n_cores = 1
steps = np.array([1,1,1,2,2,1]).reshape((-1,2))
weights = np.array([2,3,3])
downsample = 1
inputs = alignDenseSparseDTW_batch(query_list, featdir1, featdir2, outdir, n_cores, steps, weights, downsample)

Process ForkPoolWorker-1:
KeyboardInterrupt
Traceback (most recent call last):
  File "/home/jliu/ttmp/anaconda3/envs/e207project/lib/python3.7/multiprocessing/process.py", line 297, in _bootstrap
    self.run()
  File "/home/jliu/ttmp/anaconda3/envs/e207project/lib/python3.7/multiprocessing/process.py", line 99, in run
    self._target(*self._args, **self._kwargs)
  File "/home/jliu/ttmp/anaconda3/envs/e207project/lib/python3.7/multiprocessing/pool.py", line 121, in worker
    result = (True, func(*args, **kwds))
  File "/home/jliu/ttmp/anaconda3/envs/e207project/lib/python3.7/multiprocessing/pool.py", line 47, in starmapstar
    return list(itertools.starmap(args[0], args[1]))
  File "<ipython-input-4-1779d5d67e27>", line 26, in alignDenseSparseDTW
    D, B, wp = dtw_sparse_subseq(C, gaplens, steps, weights)
  File "<ipython-input-3-19e8e733278d>", line 60, in dtw_sparse_subseq
    costForStep = C[row, col] * weights[stepIndex] + D[row - steps[stepIndex][0], col - steps[stepIndex][1]

KeyboardInterrupt: 